In [4]:
# Telco Customer Churn - Data Exploration

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Load the data
df = pd.read_csv('../data/Telco_customer_churn.csv')

# Display basic information
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Dataset Shape: {df.shape}")
print(f"Number of Records: {len(df)}")
print(f"Number of Features: {len(df.columns)}")

# Display first few rows
print("\nFirst 5 rows of the dataset:")
print(df.head())

# Display column information
print("\n" + "="*60)
print("COLUMN INFORMATION")
print("="*60)

# Create a DataFrame with column information
column_info = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null Percentage': (df.isnull().sum() / len(df) * 100).values,
    'Unique Values': [df[col].nunique() for col in df.columns]
})

print(column_info.to_string())

# Check for missing values
print("\n" + "="*60)
print("MISSING VALUES ANALYSIS")
print("="*60)

missing_info = column_info[column_info['Null Count'] > 0]
if len(missing_info) > 0:
    print(missing_info.to_string())
else:
    print("No missing values found in the dataset!")

# Check for duplicate records
print("\n" + "="*60)
print("DUPLICATE RECORDS ANALYSIS")
print("="*60)

duplicates = df.duplicated().sum()
print(f"Number of duplicate records: {duplicates}")
print(f"Percentage of duplicates: {duplicates/len(df)*100:.2f}%")

# Basic statistical summary
print("\n" + "="*60)
print("NUMERICAL FEATURES STATISTICAL SUMMARY")
print("="*60)

# Select numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 0:
    print(df[numerical_cols].describe().T)

# Categorical features analysis
print("\n" + "="*60)
print("CATEGORICAL FEATURES ANALYSIS")
print("="*60)

# Select categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"\n{col}:")
    print(f"  Unique values: {unique_count}")
    if unique_count <= 10:  # Show value counts for columns with few unique values
        print(f"  Value counts:")
        print(df[col].value_counts().head())

# Target variable analysis
print("\n" + "="*60)
print("TARGET VARIABLE ANALYSIS (CHURN)")
print("="*60)

# Check which columns contain churn information
churn_cols = [col for col in df.columns if 'churn' in col.lower() or 'Churn' in col]
print(f"Churn-related columns: {churn_cols}")

for col in churn_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    if df[col].dtype in ['int64', 'float64']:
        print(f"Churn Rate: {df[col].mean()*100:.2f}%")

# Data type issues check
print("\n" + "="*60)
print("DATA TYPE ISSUES CHECK")
print("="*60)

# Check columns that might be numeric but stored as strings
potential_numeric_cols = []
for col in df.select_dtypes(include=['object']).columns:
    try:
        pd.to_numeric(df[col].replace(' ', np.nan))
        potential_numeric_cols.append(col)
    except:
        pass

if potential_numeric_cols:
    print(f"Columns that might be numeric: {potential_numeric_cols}")
    for col in potential_numeric_cols:
        print(f"\n{col} sample values:")
        print(df[col].head())
else:
    print("No obvious data type issues found.")

# Memory usage analysis
print("\n" + "="*60)
print("MEMORY USAGE ANALYSIS")
print("="*60)

memory_usage = df.memory_usage(deep=True)
total_memory = memory_usage.sum() / 1024**2  # Convert to MB

print(f"Total memory usage: {total_memory:.2f} MB")
print("\nTop 10 memory-consuming columns:")
print((memory_usage.sort_values(ascending=False).head(10) / 1024**2).to_string())

# Save exploration summary
print("\n" + "="*60)
print("SAVING EXPLORATION SUMMARY")
print("="*60)

# Create summary DataFrame
summary_data = []
for col in df.columns:
    summary_data.append({
        'Column': col,
        'Data Type': str(df[col].dtype),
        'Non-Null Count': df[col].notnull().sum(),
        'Null Count': df[col].isnull().sum(),
        'Null %': (df[col].isnull().sum() / len(df) * 100),
        'Unique Values': df[col].nunique(),
        'Sample Values': str(df[col].unique()[:3].tolist()) if df[col].nunique() > 3 else str(df[col].unique().tolist())
    })

summary_df = pd.DataFrame(summary_data)

# Save to CSV
summary_df.to_csv('../reports_2/data_exploration_summary.csv', index=False)
print("Exploration summary saved to '../reports_2/data_exploration_summary.csv'")

# Generate key insights
print("\n" + "="*60)
print("KEY INSIGHTS FROM DATA EXPLORATION")
print("="*60)

insights = f"""
DATA EXPLORATION INSIGHTS
==========================

1. DATASET SIZE:
   - Total Records: {len(df):,}
   - Total Features: {len(df.columns)}
   - Memory Usage: {total_memory:.2f} MB

2. DATA QUALITY:
   - Missing Values: {df.isnull().sum().sum()} total missing values
   - Duplicate Records: {duplicates} ({duplicates/len(df)*100:.2f}%)
   - Columns with Missing Values: {len(column_info[column_info['Null Count'] > 0])}

3. TARGET VARIABLE:
   - Churn Rate: {df['Churn Value'].mean()*100:.2f}% (if 'Churn Value' exists)
   - Churn-related columns: {churn_cols}

4. DATA TYPES:
   - Numerical Columns: {len(numerical_cols)}
   - Categorical Columns: {len(categorical_cols)}
   - Potential Data Type Issues: {len(potential_numeric_cols)} columns

5. RECOMMENDATIONS:
   - Investigate columns with many unique values for potential dimensionality reduction
   - Check data type conversions needed (e.g., 'Total Charges' might need to be numeric)
   - Analyze the distribution of key features before modeling

Next Steps:
1. Perform detailed EDA visualization in the next notebook
2. Clean and preprocess the data
3. Engineer new features
4. Build predictive models
"""

print(insights)

# Save insights to file
with open('../reports_2/data_exploration_insights.txt', 'w') as f:
    f.write(insights)

print("\nInsights saved to '../reports_2/data_exploration_insights.txt'")

DATASET OVERVIEW
Dataset Shape: (7043, 33)
Number of Records: 7043
Number of Features: 33

First 5 rows of the dataset:
   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   

                 Lat Long   Latitude   Longitude  Gender  ...        Contract  \
0  33.964131, -118.272783  33.964131 -118.272783    Male  ...  Month-to-month   
1   34.059281, -118.30742  34.059281 -118.307420  Female  ...  Month-to-month   
2  34.048013, -118.293953  34.048013 -118.293953  Female  ...  Month-to-month   
3  34.062125, -118.315709  34.062125 -118.315709  Female  ...  Month-to-month   
4  34.039224, -118.266293  34.